# 07. LLM-Assisted Interpretation of Biomedical Abstracts

## Purpose

This notebook uses a locally hosted **Qwen2.5 3B** model through **Ollama** to produce structured, source-grounded interpretations of selected PubMed abstracts. It complements the TF–IDF, LDA, VADER, and embedding analyses; it does **not** replace verification against the original scientific abstracts.

### Analysis plan

1. Confirm Python dependencies and the local Ollama model.
2. Load and validate the prepared 3-layer PubMed corpus.
3. Select a reproducible sample of **100 abstracts** (34 / 33 / 33 across the three layers), retaining the original 15 selected records.
4. Generate structured responses with a consistent prompt.
5. Save results incrementally so an interrupted run can resume.
6. Validate JSON responses and export structured results.
7. Create a manual evaluation sheet for human review.

### Important interpretation boundaries

- The source material consists of PubMed abstracts, **not actual medical-science-liaison engagement transcripts**.
- LLM outputs are exploratory assistance. They may omit details, misread scientific context, or make unsupported claims.
- A successfully returned or parsed response is a technical success, **not proof of factual accuracy**.
- Review each output against its original abstract before reporting substantive findings.
- The three corpus layers are author-defined analytical categories.

### How to run this notebook

Run the numbered steps in order on the first run. If processing is interrupted, rerun **Step 7 (LLM processing)** to resume from valid saved responses. Do not delete `llm_assisted_interpretation_outputs` before you have finished processing and evaluation.

## Step 1. Import libraries and configure paths

This step imports the packages used throughout the notebook and defines the dataset, output directory, local model name, and reproducibility settings. It does not send any text to the model.

In [1]:
# ============================================================
# STEP 1. IMPORT LIBRARIES AND CONFIGURE THE WORKFLOW
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

# Input dataset created in the earlier corpus-preparation workflow.
DATA_PATH = Path("pubmed_medical_affairs_3layer_corpus_FINAL.csv")

# All outputs from this notebook are written here.
OUTPUT_DIR = Path("llm_assisted_interpretation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Local Ollama configuration.
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
MODEL_NAME = "qwen2.5:3b"

# Reproducible sampling settings.
RANDOM_STATE = 42
TARGET_SAMPLE_SIZE = 100

# Analytical layers used in the study.
LAYERS = [
    "L1_scientific_engagement",
    "L2_safety_risk",
    "L3_comparative_critical"
]

# Output paths.
SAMPLE_PATH = OUTPUT_DIR / "llm_selected_sample_100.csv"
RAW_PATH = OUTPUT_DIR / "llm_raw_outputs.csv"
STRUCTURED_PATH = OUTPUT_DIR / "llm_structured_outputs.csv"
EVALUATION_PATH = OUTPUT_DIR / "llm_manual_evaluation.csv"

print("Libraries imported successfully.")
print("Input dataset:", DATA_PATH.resolve())
print("Output directory:", OUTPUT_DIR.resolve())
print("Configured model:", MODEL_NAME)

Libraries imported successfully.
Input dataset: C:\Users\DELL\Documents\MS thesis\pubmed_medical_affairs_3layer_corpus_FINAL.csv
Output directory: C:\Users\DELL\Documents\MS thesis\llm_assisted_interpretation_outputs
Configured model: qwen2.5:3b


## Step 2. Verify the Ollama server and model

Ollama must be running locally and the configured model must be available. If this check fails, start Ollama and confirm that `qwen2.5:3b` is installed before continuing.

In [2]:
# ============================================================
# STEP 2. CHECK OLLAMA CONNECTION AND MODEL AVAILABILITY
# ============================================================

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    response.raise_for_status()
    available_models = response.json().get("models", [])
    available_names = [model.get("name", "") for model in available_models]
except requests.RequestException as exc:
    raise RuntimeError(
        "Could not reach the local Ollama server at "
        f"{OLLAMA_BASE_URL}. Start Ollama, then rerun this step. Details: {exc}"
    ) from exc

print("Available local models:")
for name in available_names:
    print(" -", name)

if MODEL_NAME not in available_names:
    raise RuntimeError(
        f"Configured model '{MODEL_NAME}' was not found. "
        "Check the model name above or download it with Ollama, then rerun this step."
    )

print(f"\nConnection check passed. '{MODEL_NAME}' is available.")

Available local models:
 - qwen2.5:3b

Connection check passed. 'qwen2.5:3b' is available.


## Step 3. Load and validate the prepared corpus

The notebook uses the same final CSV as the preceding analyses. This cell checks required columns, reviews the layer distribution, and identifies usable non-empty abstracts. It does not modify the source CSV.

In [3]:
# ============================================================
# STEP 3. LOAD AND VALIDATE THE CORPUS
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place the CSV in the notebook's working directory or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"pmid", "title", "year", "layer", "abstract"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {sorted(missing_columns)}")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nRecords by layer in the source dataset:")
display(df["layer"].value_counts(dropna=False).to_frame("record_count"))

# Keep records from the three defined layers with non-empty abstracts.
df_valid = df.loc[
    df["layer"].isin(LAYERS)
    & df["abstract"].notna()
    & df["abstract"].astype(str).str.strip().ne("")
].copy()

# Consistent identifiers support safe matching to previously saved results.
df_valid["pmid_key"] = df_valid["pmid"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

if df_valid["pmid_key"].duplicated().any():
    duplicates = df_valid.loc[df_valid["pmid_key"].duplicated(), "pmid_key"].head().tolist()
    raise ValueError(f"Duplicate PMIDs found among valid records, e.g. {duplicates}")

print("\nValid records with non-empty abstracts:", len(df_valid))
print("Valid records per layer:")
display(df_valid["layer"].value_counts().reindex(LAYERS).to_frame("valid_records"))

Dataset shape: (21256, 6)
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'layer']

Records by layer in the source dataset:


,record_count
layer,
L1_scientific_engagement,9599
L3_comparative_critical,5849
L2_safety_risk,5808



Valid records with non-empty abstracts: 21256
Valid records per layer:


,valid_records
layer,
L1_scientific_engagement,9599
L2_safety_risk,5808
L3_comparative_critical,5849


## Step 4. Select a reproducible sample of 100 abstracts

The target sample is balanced across the three analytical layers (34 for L1, 33 for L2, and 33 for L3). The original exploratory sample of five abstracts per layer is recreated with the original random seed and retained in the expanded sample, allowing already-completed responses to be reused if their PMIDs match. The selected sample is saved for traceability.

In [4]:
# ============================================================
# STEP 4. SELECT A STRATIFIED SAMPLE OF 100 ABSTRACTS
# ============================================================

SAMPLE_COUNTS = {
    "L1_scientific_engagement": 34,
    "L2_safety_risk": 33,
    "L3_comparative_critical": 33
}

if sum(SAMPLE_COUNTS.values()) != TARGET_SAMPLE_SIZE:
    raise ValueError("Layer allocations must sum to TARGET_SAMPLE_SIZE.")

# Check that each layer has enough eligible records.
for layer, required_n in SAMPLE_COUNTS.items():
    available_n = int((df_valid["layer"] == layer).sum())
    if available_n < required_n:
        raise ValueError(
            f"Layer {layer} has {available_n} valid records; "
            f"{required_n} are required."
        )

# Recreate the original 15-record sample exactly from the valid data.
# This mirrors the first notebook version: five records per layer,
# sampled with random_state=42 within each group.
original_15 = (
    df_valid[df_valid["layer"].isin(LAYERS)]
    .groupby("layer", group_keys=False)
    .sample(n=5, random_state=42)
)

sample_parts = []
for layer_index, layer in enumerate(LAYERS):
    layer_records = df_valid[df_valid["layer"] == layer]
    retained = original_15[original_15["layer"] == layer]
    remaining_records = layer_records.drop(index=retained.index)
    extra_n = SAMPLE_COUNTS[layer] - len(retained)

    # Different, fixed seeds for the additional records in each layer.
    additional = remaining_records.sample(
        n=extra_n,
        random_state=100 + layer_index
    )
    sample_parts.append(pd.concat([retained, additional]))

sample_df = (
    pd.concat(sample_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

# Store a stable key used to resume completed generations.
sample_df["pmid_key"] = (
    sample_df["pmid"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
)

assert len(sample_df) == TARGET_SAMPLE_SIZE, "Sample must contain exactly 100 abstracts."
assert sample_df["pmid_key"].nunique() == TARGET_SAMPLE_SIZE, "Sample PMIDs must be unique."
assert sample_df["layer"].value_counts().to_dict() == SAMPLE_COUNTS, "Unexpected layer allocation."

sample_df.drop(columns=["pmid_key"]).to_csv(SAMPLE_PATH, index=False)

print("Selected sample size:", len(sample_df))
print("Original 15 records retained:", len(original_15))
print("\nSample allocation by layer:")
display(sample_df["layer"].value_counts().reindex(LAYERS).to_frame("abstract_count"))
print("\nSample saved to:", SAMPLE_PATH)
display(sample_df[["pmid", "title", "year", "layer", "abstract"]].head(5))

Selected sample size: 100
Original 15 records retained: 15

Sample allocation by layer:


,abstract_count
layer,
L1_scientific_engagement,34
L2_safety_risk,33
L3_comparative_critical,33



Sample saved to: llm_assisted_interpretation_outputs\llm_selected_sample_100.csv


,pmid,title,year,layer,abstract
0,41262834,Effects of Pregnancy on Oral Health: A Narrati...,2025.0,L3_comparative_critical,Pregnancy induces significant physiological an...
1,41412207,Feasibility of 5-Aminolevulenic Acid Fluoresce...,2025.0,L2_safety_risk,BACKGROUND: Intracranial lymphomas are rare tu...
2,41259711,Resurgence of Acquired Syphilitic Uveitis: Cha...,2025.0,L3_comparative_critical,PURPOSE: Syphilitic uveitis (SU) is a re-emerg...
3,41316485,Advances in the characterization of in vitro-g...,2025.0,L2_safety_risk,"BACKGROUND: Red blood cells (RBCs), essential ..."
4,41014098,A scoping review and comprehensive needs asses...,2025.0,L2_safety_risk,Mild cognitive impairment (MCI) is a high-risk...


## Step 5. Define the local LLM request function

This function sends one prompt to Ollama and returns the raw model response. JSON output mode is requested to improve structured-output consistency. A short test confirms that a generation request works before the 100-record run.

In [5]:
# ============================================================
# STEP 5. DEFINE AND TEST THE OLLAMA GENERATION FUNCTION
# ============================================================

def ask_local_llm(prompt, timeout_seconds=300):
    """Send a prompt to the local Ollama model and return its text response."""
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0
        }
    }

    response = requests.post(
        OLLAMA_GENERATE_URL,
        json=payload,
        timeout=timeout_seconds
    )
    response.raise_for_status()

    response_json = response.json()
    output = response_json.get("response")
    if not isinstance(output, str) or not output.strip():
        raise ValueError("Ollama returned an empty or unrecognised response.")

    return output.strip()

# Small technical test only; no study abstract is included in this request.
test_output = ask_local_llm(
    'Return a JSON object with one key named "status" and value "LOCAL MODEL CONNECTED".'
)
print("Test response:", test_output)

try:
    test_parsed = json.loads(test_output)
except json.JSONDecodeError as exc:
    raise RuntimeError("The Ollama test response was not valid JSON.") from exc

print("Local model request test passed:", test_parsed)

Test response: {
  "status": "LOCAL MODEL CONNECTED"
}
Local model request test passed: {'status': 'LOCAL MODEL CONNECTED'}


## Step 6. Define the consistent extraction prompt

The same prompt is used for all abstracts. It asks for the research objective, main findings, reported efficacy and safety information, explicitly stated limitations, future research, and a concise summary. Missing information must be labelled as not reported rather than inferred.

In [6]:
# ============================================================
# STEP 6. DEFINE THE LLM EXTRACTION PROMPT
# ============================================================

PROMPT_VERSION = "v1.0"

OUTPUT_FIELDS = [
    "research_objective",
    "main_findings",
    "efficacy_information",
    "safety_information",
    "explicit_limitations",
    "future_research",
    "summary"
]


def build_prompt(abstract):
    """Build a consistent, source-grounded prompt for one abstract."""
    return f"""You are assisting with an academic analysis of biomedical literature.

Analyse the abstract below. Use only information explicitly supported by the abstract.

Return one valid JSON object with exactly these fields:
{{
  "research_objective": "Main objective or research question",
  "main_findings": "Principal findings reported",
  "efficacy_information": "Reported efficacy information, or Not reported in the abstract",
  "safety_information": "Reported safety information, or Not reported in the abstract",
  "explicit_limitations": "Limitations explicitly stated by the authors, or Not explicitly reported in the abstract",
  "future_research": "Recommendations for future research, or Not reported in the abstract",
  "summary": "Concise summary in no more than three sentences"
}}

Rules:
- Use the abstract as the only source of evidence.
- Do not invent results, statistics, limitations, or conclusions.
- Distinguish findings from the authors' interpretations where possible.
- A future-research recommendation is not automatically an explicit limitation.
- Do not infer that a treatment is effective or safe beyond what is reported.
- If information is absent, state that it is not reported in the abstract.
- Return JSON only, without Markdown fences or additional commentary.

ABSTRACT:
{abstract}
"""

print("Prompt created. Version:", PROMPT_VERSION)
print("Output fields:", OUTPUT_FIELDS)

Prompt created. Version: v1.0
Output fields: ['research_objective', 'main_findings', 'efficacy_information', 'safety_information', 'explicit_limitations', 'future_research', 'summary']


## Step 7. Generate responses with checkpointing and resume support

This is the longest step. It checks the existing raw-results CSV and reuses responses for selected PMIDs only when the saved response is valid JSON and has no recorded request error. Results are saved after each newly attempted record. If the notebook is interrupted, rerun this step; successful saved responses will be reused and missing/invalid responses retried.

In [7]:
# ============================================================
# STEP 7. RUN LLM GENERATION WITH CHECKPOINTING
# ============================================================


def normalise_pmid(value):
    """Normalise a PMID to a stable string for joins and checkpointing."""
    text_value = str(value).strip()
    if text_value.endswith(".0"):
        text_value = text_value[:-2]
    return text_value


def is_valid_json_object(value):
    """Return True when value contains a parseable JSON object."""
    if not isinstance(value, str) or not value.strip():
        return False
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return False
    return isinstance(parsed, dict)

sample_pmid_order = [normalise_pmid(pmid) for pmid in sample_df["pmid"]]
selected_pmid_set = set(sample_pmid_order)
results_by_pmid = {}

# Recover valid previously saved responses for PMIDs in this sample.
if RAW_PATH.exists():
    previous_df = pd.read_csv(RAW_PATH, dtype={"pmid": "string"})
    for _, previous in previous_df.iterrows():
        key = normalise_pmid(previous.get("pmid", ""))
        if key not in selected_pmid_set:
            continue

        raw_output = previous.get("llm_output")
        error_value = previous.get("error")
        has_error = pd.notna(error_value) and str(error_value).strip() != ""

        if has_error or not is_valid_json_object(raw_output):
            continue

        results_by_pmid[key] = {
            "pmid": previous.get("pmid"),
            "title": previous.get("title"),
            "year": previous.get("year"),
            "layer": previous.get("layer"),
            "abstract": previous.get("abstract"),
            "llm_output": raw_output,
            "error": None,
            "prompt_version": PROMPT_VERSION
        }

print("Valid cached responses recovered:", len(results_by_pmid))
print("Abstracts to process or retry:", len(sample_df) - len(results_by_pmid))

# Process each selected abstract in the fixed sample order.
for run_position, (_, row) in enumerate(sample_df.iterrows(), start=1):
    pmid = row["pmid"]
    key = normalise_pmid(pmid)

    if key in results_by_pmid:
        print(f"[{run_position}/{len(sample_df)}] Reusing valid response | PMID {pmid}")
        continue

    abstract = str(row["abstract"])
    record = {
        "pmid": pmid,
        "title": row["title"],
        "year": row["year"],
        "layer": row["layer"],
        "abstract": abstract,
        "llm_output": None,
        "error": None,
        "prompt_version": PROMPT_VERSION
    }

    try:
        prompt = build_prompt(abstract)
        record["llm_output"] = ask_local_llm(prompt)
        print(f"[{run_position}/{len(sample_df)}] Request completed | {row['layer']} | PMID {pmid}")
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
        print(f"[{run_position}/{len(sample_df)}] ERROR | PMID {pmid} | {record['error']}")

    results_by_pmid[key] = record

    # Save after each attempt so work is not lost if a later request fails.
    checkpoint_records = [
        results_by_pmid[p]
        for p in sample_pmid_order
        if p in results_by_pmid
    ]
    pd.DataFrame(checkpoint_records).to_csv(RAW_PATH, index=False)

    # Brief pause reduces the chance of overwhelming the local service.
    time.sleep(0.2)

# Build the in-memory table in the same order as the selected sample.
llm_results_df = pd.DataFrame([
    results_by_pmid[p]
    for p in sample_pmid_order
    if p in results_by_pmid
])
llm_results_df.to_csv(RAW_PATH, index=False)

print("\nGeneration pass finished.")
print("Records represented in results:", len(llm_results_df))
print("Recorded request errors:", int(llm_results_df["error"].notna().sum()))
print("Raw checkpoint file:", RAW_PATH)

if len(llm_results_df) != TARGET_SAMPLE_SIZE:
    print("WARNING: The results table does not yet contain all 100 selected abstracts.")
    print("Review the checkpoint file and rerun Step 7 if required.")

Valid cached responses recovered: 15
Abstracts to process or retry: 85
[1/100] Request completed | L3_comparative_critical | PMID 41262834
[2/100] Request completed | L2_safety_risk | PMID 41412207
[3/100] Reusing valid response | PMID 41259711
[4/100] Request completed | L2_safety_risk | PMID 41316485
[5/100] Request completed | L2_safety_risk | PMID 41014098
[6/100] Request completed | L2_safety_risk | PMID 41359090
[7/100] Request completed | L1_scientific_engagement | PMID 41360383
[8/100] Request completed | L3_comparative_critical | PMID 41474630
[9/100] Request completed | L1_scientific_engagement | PMID 41111923
[10/100] Reusing valid response | PMID 41418718
[11/100] Request completed | L1_scientific_engagement | PMID 41147906
[12/100] Request completed | L1_scientific_engagement | PMID 41362531
[13/100] Request completed | L3_comparative_critical | PMID 41451228
[14/100] Request completed | L1_scientific_engagement | PMID 41309192
[15/100] Request completed | L3_comparative_c

## Step 8. Validate and parse the model responses

The raw model response is retained unchanged for auditability. This step attempts to parse each response as a JSON object and records which responses are suitable for structured extraction. A parseable response is not necessarily factually correct.

In [8]:
# ============================================================
# STEP 8. VALIDATE AND PARSE JSON RESPONSES
# ============================================================


def parse_llm_output(output):
    """Parse a raw response as JSON; return a dict or None."""
    if not isinstance(output, str) or not output.strip():
        return None
    try:
        parsed = json.loads(output)
    except json.JSONDecodeError:
        return None
    return parsed if isinstance(parsed, dict) else None

llm_results_df["parsed_output"] = llm_results_df["llm_output"].apply(parse_llm_output)
llm_results_df["json_parse_success"] = llm_results_df["parsed_output"].apply(lambda value: isinstance(value, dict))

print("Records in results:", len(llm_results_df))
print("Valid JSON objects:", int(llm_results_df["json_parse_success"].sum()))
print("Responses requiring retry or review:", int((~llm_results_df["json_parse_success"]).sum()))
print("Request errors:", int(llm_results_df["error"].notna().sum()))

print("\nJSON validity by layer:")
display(
    llm_results_df.groupby("layer")["json_parse_success"]
    .agg(record_count="size", valid_json_count="sum")
    .assign(valid_json_rate=lambda x: (x["valid_json_count"] / x["record_count"]).round(3))
)

Records in results: 100
Valid JSON objects: 99
Responses requiring retry or review: 1
Request errors: 1

JSON validity by layer:


,record_count,valid_json_count,valid_json_rate
layer,,,
L1_scientific_engagement,34,34,1.00
L2_safety_risk,33,32,0.97
L3_comparative_critical,33,33,1.00


## Step 9. Save raw and structured outputs

The raw CSV retains the original abstract and model response. The structured CSV places the requested JSON fields into separate columns. Missing fields are not inferred; if a response failed to parse, its structured fields remain blank and should be reviewed or regenerated.

In [9]:
# ============================================================
# STEP 9. EXPORT RAW AND STRUCTURED OUTPUTS
# ============================================================

# Keep parsed_output out of the raw CSV because it is a Python object;
# llm_output remains the auditable, original model response.
raw_export_columns = [
    "pmid", "title", "year", "layer", "abstract",
    "llm_output", "error", "prompt_version", "json_parse_success"
]
raw_export_df = llm_results_df.copy()
for column in raw_export_columns:
    if column not in raw_export_df.columns:
        raw_export_df[column] = None
raw_export_df[raw_export_columns].to_csv(RAW_PATH, index=False)

structured_records = []
for _, row in llm_results_df.iterrows():
    parsed = row.get("parsed_output")
    record = {
        "pmid": row.get("pmid"),
        "title": row.get("title"),
        "year": row.get("year"),
        "layer": row.get("layer"),
        "error": row.get("error"),
        "json_parse_success": bool(row.get("json_parse_success", False)),
        "prompt_version": row.get("prompt_version", PROMPT_VERSION)
    }

    for field in OUTPUT_FIELDS:
        record[field] = parsed.get(field, "") if isinstance(parsed, dict) else ""

    structured_records.append(record)

structured_df = pd.DataFrame(structured_records)
structured_df.to_csv(STRUCTURED_PATH, index=False)

print("Raw responses saved to:", RAW_PATH)
print("Structured outputs saved to:", STRUCTURED_PATH)
print("Structured records:", len(structured_df))
print("Valid JSON responses:", int(structured_df["json_parse_success"].sum()))

Raw responses saved to: llm_assisted_interpretation_outputs\llm_raw_outputs.csv
Structured outputs saved to: llm_assisted_interpretation_outputs\llm_structured_outputs.csv
Structured records: 100
Valid JSON responses: 99


## Step 10. Inspect a subset of structured outputs

This is a quick visual review, not a formal evaluation. Read several records from each layer and compare the generated fields to the original abstract. Check for unsupported details, omissions, conflation of limitations with future work, and overstatements of efficacy or safety.

In [10]:
# ============================================================
# STEP 10. INSPECT STRUCTURED OUTPUTS
# ============================================================

pd.set_option("display.max_colwidth", 150)

inspection_columns = [
    "pmid", "layer", "research_objective", "main_findings",
    "efficacy_information", "safety_information",
    "explicit_limitations", "future_research", "summary",
    "json_parse_success", "error"
]

print("Structured output columns:")
print(structured_df.columns.tolist())

display(structured_df[inspection_columns].head(10))

print("\nRecord counts by layer:")
display(structured_df["layer"].value_counts().reindex(LAYERS).to_frame("record_count"))

Structured output columns:
['pmid', 'title', 'year', 'layer', 'error', 'json_parse_success', 'prompt_version', 'research_objective', 'main_findings', 'efficacy_information', 'safety_information', 'explicit_limitations', 'future_research', 'summary']


,pmid,layer,research_objective,main_findings,efficacy_information,safety_information,explicit_limitations,future_research,summary,json_parse_success,error
0,41262834,L3_comparative_critical,To explore the factors responsible for oral health conditions during pregnancy and to recommend comprehensive management strategies.,"Changes in immunological response, oral environment, increased levels of steroid hormones, and nutritional deficits contribute to oral health cond...",Not reported in the abstract,The second trimester is recommended for dental procedures to ensure optimal maternal and fetal health.,Not explicitly reported in the abstract,Develop targeted interventions and innovative solutions for delivering oral health education and preventive care to pregnant women.,"The abstract discusses the impact of pregnancy on oral health, highlighting the need for comprehensive management strategies, including prevention...",True,None
1,41412207,L2_safety_risk,To evaluate the diagnostic performance of 5-ALA-guided biopsy in adult patients with radiologically suspected intracranial lymphoma and to assess ...,"5-ALA fluorescence was observed in 77.6% of 18 studies (170 patients), with 78% pooled sensitivity and 22% false-negative rate. In the institution...","5-ALA fluorescence serves as a useful adjunct for intracranial lymphoma biopsy, enhancing sampling efficiency and procedural safety. However, fluo...",Not reported in the abstract.,"Fluorescence alone cannot ensure diagnostic adequacy, and frozen section confirmation remains necessary.",Recommendations for future research include further studies to determine the optimal corticosteroid use and to explore the potential of 5-ALA fluo...,"The study found that 5-ALA fluorescence is useful for guiding biopsy in intracranial lymphoma, but its efficacy is not absolute. Corticosteroid us...",True,None
2,41259711,L3_comparative_critical,"To characterize the clinical, imaging features, treatment modalities, and outcomes of syphilitic uveitis (SU) at a tertiary-care center in North I...","The study included 26 patients (44 eyes) with syphilitic uveitis, predominantly male with a mean age of 37.9 ± 12.1 years. The majority of patient...",Definitive treatment with intravenous ceftriaxone or aqueous penicillin with adjunctive corticosteroids was effective in achieving resolution in 9...,"Corticosteroids were empirically used in 38.5% of cases, and aqueous penicillin was used in definitive treatment.",Not explicitly reported,Recommendations for future research include further evaluation of aqueous penicillin as a treatment modality.,"The study found syphilitic uveitis to be prevalent, especially in the post-COVID era, with favorable outcomes achieved with intravenous ceftriaxon...",True,None
3,41316485,L2_safety_risk,To integrate multi-omics platforms to comprehensively profile the developmental and functional states of in vitro-generated red blood cells (RBCs)...,"The review highlights the challenges in recapitulating erythropoiesis in vitro, emphasizing the need for a systems-level perspective that integrat...",Not reported in the abstract,Not reported in the abstract,"The review outlines the challenges in characterizing in vitro-generated RBCs, including the difficulty in recapitulating erythropoiesis in vitro a...","The review proposes a translational framework that integrates multi-omics data into the development and evaluation of in vitro-generated RBCs, and...",The review emphasizes the need for a comprehensive approach to characterize in vitro-generated red blood cells (RBCs) using multi-omics platforms....,True,None
4,41014098,L2_safety_risk,"Identify cognitive, physical, and psychosocial challenges experienced by MCI patients and explore the needs, preferences, barriers, and facilitato...","Highlights challenges such as memory loss, fatigue, social withdrawal, and a desire for meaningful, safe, and structured activities. Identifies ba...",Not reported in the abstract,Not reported in the 


Record counts by layer:


,record_count
layer,
L1_scientific_engagement,34
L2_safety_risk,33
L3_comparative_critical,33


## Step 11. Create the manual evaluation sheet

Review each generated interpretation against the original abstract. Use the following rubric consistently:

| Criterion | 0 | 1 | 2 |
|---|---|---|---|
| Factual consistency | Major unsupported or incorrect statements | Minor error or questionable inference | Claims supported by abstract |
| Relevance | Mostly irrelevant or misses the main purpose | Partly relevant | Directly addresses the abstract |
| Information extraction | Important findings omitted or misrepresented | Some key details captured | Key objective/findings captured accurately |
| Limitation identification | Invented/misclassified limitation or clear omission | Partly correct or ambiguous | Explicit limitations reported accurately; absence represented cautiously |
| Summary quality | Misleading/inaccurate | Partly useful but incomplete | Concise and faithful to the abstract |

Each criterion is scored 0–2, for a maximum of 10 per abstract. Use the comments column to note evidence, errors, omissions, or reasons for ambiguous scores. Scores are reviewer judgements; they are not a general model-accuracy estimate. The code preserves previous scores and comments for PMIDs that overlap if the evaluation sheet already exists.

In [11]:
# ============================================================
# STEP 11. CREATE THE MANUAL EVALUATION SHEET
# ============================================================

score_columns = [
    "factual_consistency_score",
    "relevance_score",
    "information_extraction_score",
    "limitation_identification_score",
    "summary_quality_score"
]
comment_column = "reviewer_comments"

evaluation_columns = [
    "pmid", "title", "year", "layer", "abstract",
    "research_objective", "main_findings", "efficacy_information",
    "safety_information", "explicit_limitations", "future_research",
    "summary", "json_parse_success", "error"
]

evaluation_df = structured_df.merge(
    llm_results_df[["pmid", "abstract"]].drop_duplicates("pmid"),
    on="pmid", how="left", suffixes=("", "_raw")
)

# Ensure all evaluation fields exist and maintain a useful column order.
for column in evaluation_columns:
    if column not in evaluation_df.columns:
        evaluation_df[column] = ""
evaluation_df = evaluation_df[evaluation_columns].copy()

for column in score_columns:
    evaluation_df[column] = pd.NA
evaluation_df[comment_column] = ""

# If an evaluation sheet already exists, carry forward completed ratings
# and comments by PMID rather than overwriting human work.
if EVALUATION_PATH.exists():
    try:
        previous_eval = pd.read_csv(EVALUATION_PATH, dtype={"pmid": "string"})
        previous_eval["pmid_key"] = previous_eval["pmid"].map(normalise_pmid)
        evaluation_df["pmid_key"] = evaluation_df["pmid"].map(normalise_pmid)

        keep_cols = ["pmid_key"] + [
            col for col in score_columns + [comment_column]
            if col in previous_eval.columns
        ]
        previous_ratings = previous_eval[keep_cols].drop_duplicates("pmid_key")
        evaluation_df = evaluation_df.merge(
            previous_ratings, on="pmid_key", how="left", suffixes=("", "_old")
        )
        for column in score_columns + [comment_column]:
            old_column = f"{column}_old"
            if old_column in evaluation_df.columns:
                evaluation_df[column] = evaluation_df[old_column].combine_first(evaluation_df[column])
                evaluation_df.drop(columns=[old_column], inplace=True)
        evaluation_df.drop(columns=["pmid_key"], inplace=True)
    except Exception as exc:
        print("Warning: prior evaluation sheet could not be merged:", exc)
        if "pmid_key" in evaluation_df.columns:
            evaluation_df.drop(columns=["pmid_key"], inplace=True)

# Put the editable rating/comment columns last for ease of manual review.
evaluation_df = evaluation_df[
    evaluation_columns + score_columns + [comment_column]
]
evaluation_df.to_csv(EVALUATION_PATH, index=False)

print("Manual evaluation sheet saved to:", EVALUATION_PATH)
print("Rows available for evaluation:", len(evaluation_df))
display(evaluation_df.head(5))

Manual evaluation sheet saved to: llm_assisted_interpretation_outputs\llm_manual_evaluation.csv
Rows available for evaluation: 100


,pmid,title,year,layer,abstract,research_objective,main_findings,efficacy_information,safety_information,explicit_limitations,future_research,summary,json_parse_success,error,factual_consistency_score,relevance_score,information_extraction_score,limitation_identification_score,summary_quality_score,reviewer_comments
0,41262834,Effects of Pregnancy on Oral Health: A Narrative Review.,2025.0,L3_comparative_critical,Pregnancy induces significant physiological and hormonal changes that can affect oral health. These changes can worsen existing oral conditions or...,To explore the factors responsible for oral health conditions during pregnancy and to recommend comprehensive management strategies.,"Changes in immunological response, oral environment, increased levels of steroid hormones, and nutritional deficits contribute to oral health cond...",Not reported in the abstract,The second trimester is recommended for dental procedures to ensure optimal maternal and fetal health.,Not explicitly reported in the abstract,Develop targeted interventions and innovative solutions for delivering oral health education and preventive care to pregnant women.,"The abstract discusses the impact of pregnancy on oral health, highlighting the need for comprehensive management strategies, including prevention...",True,None,<NA>,<NA>,<NA>,<NA>,<NA>,
1,41412207,Feasibility of 5-Aminolevulenic Acid Fluorescence for Diagnostic Guidance in Intracranial Lymphoma Surgery: A Case Series and Systematic Review of...,2025.0,L2_safety_risk,"BACKGROUND: Intracranial lymphomas are rare tumors of the central nervous system, predominantly affecting elderly patients. While 5-aminolevulinic...",To evaluate the diagnostic performance of 5-ALA-guided biopsy in adult patients with radiologically suspected intracranial lymphoma and to assess ...,"5-ALA fluorescence was observed in 77.6% of 18 studies (170 patients), with 78% pooled sensitivity and 22% false-negative rate. In the institution...","5-ALA fluorescence serves as a useful adjunct for intracranial lymphoma biopsy, enhancing sampling efficiency and procedural safety. However, fluo...",Not reported in the abstract.,"Fluorescence alone cannot ensure diagnostic adequacy, and frozen section confirmation remains necessary.",Recommendations for future research include further studies to determine the optimal corticosteroid use and to explore the potential of 5-ALA fluo...,"The study found that 5-ALA fluorescence is useful for guiding biopsy in intracranial lymphoma, but its efficacy is not absolute. Corticosteroid us...",True,None,<NA>,<NA>,<NA>,<NA>,<NA>,
2,41259711,"Resurgence of Acquired Syphilitic Uveitis: Characteristics, Challenges and Outcomes.",2025.0,L3_comparative_critical,"PURPOSE: Syphilitic uveitis (SU) is a re-emerging cause of intraocular inflammation. This study aimed to characterize the clinical, imaging featur...","To characterize the clinical, imaging features, treatment modalities, and outcomes of syphilitic uveitis (SU) at a tertiary-care center in North I...","The study included 26 patients (44 eyes) with syphilitic uveitis, predominantly male with a mean age of 37.9 ± 12.1 years. The majority of patient...",Definitive treatment with intravenous ceftriaxone or aqueous penicillin with adjunctive corticosteroids was effective in achieving resolution in 9...,"Corticosteroids were empirically used in 38.5% of cases, and aqueous penicillin was used in definitive treatment.",Not explicitly reported,Recommendations for future research include further evaluation of aqueous penicillin as a treatment modality.,"The study found syphilitic uveitis to be prevalent, especially in the post-COVID era, with favorable outcomes achieved with intravenous ceftriaxon...",True,None,<NA>,<NA>,<NA>,<NA>,<NA>,
3,41316485,Advances in the characterization of in vitro-generated red blood cells: from biophysical properties to functional applications.,2025.0,L2_safety_risk,"BACKGROUND: Red blood cells (RBCs), essential for oxygen 

## Step 12. Final processing summary and output inventory

Run this final check after the previous steps. Confirm the sample size, request errors, JSON parsing, and output files before starting the manual evaluation. If any response has an error or invalid JSON, do not count it as a valid structured interpretation until it has been regenerated or reviewed.

In [12]:
# ============================================================
# STEP 12. FINAL WORKFLOW SUMMARY
# ============================================================

summary = {
    "selected_abstracts": len(sample_df),
    "results_rows": len(llm_results_df),
    "successful_json_responses": int(llm_results_df["json_parse_success"].sum()),
    "responses_requiring_review": int((~llm_results_df["json_parse_success"]).sum()),
    "request_errors": int(llm_results_df["error"].notna().sum()),
}

print("LLM analysis summary:")
for key, value in summary.items():
    print(f" - {key}: {value}")

print("\nFiles produced:")
for path in [SAMPLE_PATH, RAW_PATH, STRUCTURED_PATH, EVALUATION_PATH]:
    print(f" - {path} | exists={path.exists()}" + (f" | bytes={path.stat().st_size}" if path.exists() else ""))

print("\nNext action: complete manual scoring in llm_manual_evaluation.csv and verify all scored claims against the source abstracts.")

LLM analysis summary:
 - selected_abstracts: 100
 - results_rows: 100
 - successful_json_responses: 99
 - responses_requiring_review: 1
 - request_errors: 1

Files produced:
 - llm_assisted_interpretation_outputs\llm_selected_sample_100.csv | exists=True | bytes=183582
 - llm_assisted_interpretation_outputs\llm_raw_outputs.csv | exists=True | bytes=322797
 - llm_assisted_interpretation_outputs\llm_structured_outputs.csv | exists=True | bytes=137194
 - llm_assisted_interpretation_outputs\llm_manual_evaluation.csv | exists=True | bytes=303479

Next action: complete manual scoring in llm_manual_evaluation.csv and verify all scored claims against the source abstracts.
